In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/iot/iot device name/network")
MEID_DIR = BASE_DIR / "Meid20"

PACKET_DIR = MEID_DIR / "packet_csvs"
FLOW_DIR   = MEID_DIR / "flow_csvs"
MERGE_DIR  = MEID_DIR / "merged"
MODEL_DIR  = MEID_DIR / "models"

for d in [MEID_DIR, PACKET_DIR, FLOW_DIR, MERGE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Created:")
print(MEID_DIR)
print(PACKET_DIR)
print(FLOW_DIR)
print(MERGE_DIR)
print(MODEL_DIR)

Created:
/content/drive/MyDrive/iot/iot device name/network/Meid20
/content/drive/MyDrive/iot/iot device name/network/Meid20/packet_csvs
/content/drive/MyDrive/iot/iot device name/network/Meid20/flow_csvs
/content/drive/MyDrive/iot/iot device name/network/Meid20/merged
/content/drive/MyDrive/iot/iot device name/network/Meid20/models


In [5]:
import pandas as pd

DEVICE_MAP_PATH = BASE_DIR / "Okui22" / "device_map.csv"

device_map = pd.read_csv(DEVICE_MAP_PATH)
device_map.columns = [c.strip().lower() for c in device_map.columns]

display(device_map.head())
print(device_map.columns.tolist())

assert "ip" in device_map.columns
assert "service_name" in device_map.columns

,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3


['ip', 'best_mac', 'service_name']


In [6]:
device_map = device_map.dropna(subset=["ip", "service_name"]).copy()
device_map["ip"] = device_map["ip"].astype(str).str.strip()
device_map["service_name"] = device_map["service_name"].astype(str).str.strip()

ip_to_device = (
    device_map.drop_duplicates(subset=["ip"])
              .set_index("ip")["service_name"]
              .to_dict()
)

print("number of mapped IPs:", len(ip_to_device))
print(ip_to_device)

number of mapped IPs: 12
{'192.168.1.133': 'service_7', '192.168.1.146': 'service_2', '192.168.1.152': 'service_1', '192.168.1.17': 'service_5', '192.168.1.190': 'service_3', '192.168.1.191': 'service_5', '192.168.1.192': 'service_2', '192.168.1.193': 'service_2', '192.168.1.194': 'service_2', '192.168.1.250': 'service_6', '192.168.1.30': 'service_5', '192.168.1.79': 'service_4'}


In [7]:
import subprocess

def pcap_to_packet_csv(pcap_path, out_csv):
    cmd = [
        "tshark",
        "-r", str(pcap_path),
        "-Y", "ip",
        "-T", "fields",
        "-E", "header=y",
        "-E", "separator=,",
        "-E", "quote=d",
        "-E", "occurrence=f",
        "-e", "frame.time_epoch",
        "-e", "ip.src",
        "-e", "ip.dst",
        "-e", "ip.proto",
        "-e", "tcp.srcport",
        "-e", "tcp.dstport",
        "-e", "udp.srcport",
        "-e", "udp.dstport",
        "-e", "frame.len",
        "-e", "tcp.flags",
        "-e", "ip.dsfield",
    ]
    with open(out_csv, "w") as f:
        subprocess.run(cmd, stdout=f, check=True)

test_pcap = BASE_DIR / "normal_2.pcap"
test_csv  = PACKET_DIR / "normal_2_packets.csv"

pcap_to_packet_csv(test_pcap, test_csv)
print("saved:", test_csv)

saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/packet_csvs/normal_2_packets.csv


In [8]:
pkt = pd.read_csv(test_csv)
display(pkt.head())
print(pkt.shape)
print(pkt.columns.tolist())

,frame.time_epoch,ip.src,ip.dst,ip.proto,tcp.srcport,tcp.dstport,udp.srcport,udp.dstport,frame.len,tcp.flags,ip.dsfield
0,1.554230e+09,192.168.1.152,3.122.49.24,6,52976.0,1883.0,NaN,NaN,197,0x0018,0x02
1,1.554230e+09,192.168.1.194,192.168.1.190,1,NaN,NaN,NaN,NaN,100,NaN,0x00
2,1.554230e+09,192.168.1.190,192.168.1.194,1,NaN,NaN,NaN,NaN,100,NaN,0x00
3,1.554230e+09,192.168.1.152,192.168.1.195,6,1880.0,49773.0,NaN,NaN,929,0x0018,0x00
4,1.554230e+09,192.168.1.195,192.168.1.152,6,49773.0,1880.0,NaN,NaN,62,0x0010,0x00


(328349, 11)
['frame.time_epoch', 'ip.src', 'ip.dst', 'ip.proto', 'tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport', 'frame.len', 'tcp.flags', 'ip.dsfield']


In [9]:
import numpy as np
import ipaddress
from functools import reduce
import operator

def to_num(x):
    return pd.to_numeric(x, errors="coerce")

def hex_or_dec_to_int(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    try:
        if s.startswith("0x") or s.startswith("0X"):
            return int(s, 16)
        return int(float(s))
    except:
        return np.nan

def is_private_ipv4(ip):
    try:
        obj = ipaddress.ip_address(str(ip))
        return obj.version == 4 and obj.is_private
    except:
        return False

def ipv4_to_net24(ip):
    try:
        obj = ipaddress.ip_address(str(ip))
        if obj.version != 4:
            return "non_ipv4"
        parts = str(obj).split(".")
        return ".".join(parts[:3]) + ".0/24"
    except:
        return "unknown"

def bitwise_or_series(s):
    vals = [int(v) for v in s.dropna().tolist()]
    if len(vals) == 0:
        return 0
    return reduce(operator.or_, vals, 0)

In [10]:
def packet_csv_to_meid20_flow_csv(packet_csv_path, flow_csv_path, ip_to_device):
    df = pd.read_csv(packet_csv_path, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    # 统一列名
    rename_map = {
        "frame.time_epoch": "time",
        "ip.src": "src_ip",
        "ip.dst": "dst_ip",
        "ip.proto": "protocol",
        "tcp.srcport": "tcp_srcport",
        "tcp.dstport": "tcp_dstport",
        "udp.srcport": "udp_srcport",
        "udp.dstport": "udp_dstport",
        "frame.len": "frame_len",
        "tcp.flags": "tcp_flags",
        "ip.dsfield": "src_tos",
    }
    df = df.rename(columns=rename_map)

    # 数值转换
    df["time"] = df["time"].apply(to_num)
    df["protocol"] = df["protocol"].apply(to_num)
    df["frame_len"] = df["frame_len"].apply(to_num)
    df["tcp_flags"] = df["tcp_flags"].apply(hex_or_dec_to_int)
    df["src_tos"] = df["src_tos"].apply(hex_or_dec_to_int)

    for c in ["tcp_srcport", "tcp_dstport", "udp_srcport", "udp_dstport"]:
        df[c] = df[c].apply(to_num)

    # 合并 TCP/UDP 端口
    df["src_port"] = df["tcp_srcport"].fillna(df["udp_srcport"])
    df["dst_port"] = df["tcp_dstport"].fillna(df["udp_dstport"])

    # 去掉核心字段缺失
    df = df.dropna(subset=["time", "src_ip", "dst_ip", "protocol", "frame_len"]).copy()

    # 只保留 device_map 里已知 IoT IP
    df["src_ip"] = df["src_ip"].astype(str)
    df["dst_ip"] = df["dst_ip"].astype(str)
    df = df[df["src_ip"].isin(ip_to_device.keys())].copy()

    # 只保留发往外部公网的流
    df = df[~df["dst_ip"].apply(is_private_ipv4)].copy()

    # 没有端口的流先去掉（这样更贴近 Meid20）
    df = df.dropna(subset=["src_port", "dst_port"]).copy()

    # 加标签
    df["device_name"] = df["src_ip"].map(ip_to_device)
    df["ipv4_dst_net"] = df["dst_ip"].apply(ipv4_to_net24)

    # flow key
    flow_keys = ["src_ip", "dst_ip", "protocol", "src_port", "dst_port", "device_name", "ipv4_dst_net"]

    flow_df = (
        df.groupby(flow_keys, dropna=False)
          .agg(
              first_switched=("time", "min"),
              last_switched=("time", "max"),
              in_bytes=("frame_len", "sum"),
              in_pkts=("frame_len", "size"),
              src_tos=("src_tos", lambda s: s.dropna().iloc[0] if s.dropna().shape[0] > 0 else 0),
              tcp_flags=("tcp_flags", bitwise_or_series),
          )
          .reset_index()
    )

    flow_df["flow_duration"] = flow_df["last_switched"] - flow_df["first_switched"]

    # 整理列顺序
    cols = [
        "device_name",
        "src_ip",
        "dst_ip",
        "ipv4_dst_net",
        "protocol",
        "src_port",
        "dst_port",
        "src_tos",
        "tcp_flags",
        "first_switched",
        "last_switched",
        "flow_duration",
        "in_bytes",
        "in_pkts",
    ]
    flow_df = flow_df[cols].copy()

    flow_df.to_csv(flow_csv_path, index=False)
    return flow_df

In [11]:
test_flow_csv = FLOW_DIR / "normal_2_meid20_flows.csv"
flow_df = packet_csv_to_meid20_flow_csv(test_csv, test_flow_csv, ip_to_device)

print("saved:", test_flow_csv)
display(flow_df.head())
print(flow_df.shape)

saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/flow_csvs/normal_2_meid20_flows.csv


,device_name,src_ip,dst_ip,ipv4_dst_net,protocol,src_port,dst_port,src_tos,tcp_flags,first_switched,last_switched,flow_duration,in_bytes,in_pkts
0,service_7,192.168.1.133,224.0.0.251,224.0.0.0/24,17,5353.0,5353.0,0,0,1.554232e+09,1.554239e+09,7413.435819,18649,170
1,service_7,192.168.1.133,239.255.255.250,239.255.255.0/24,17,49788.0,1900.0,0,0,1.554236e+09,1.554236e+09,0.000000,169,1
2,service_7,192.168.1.133,239.255.255.250,239.255.255.0/24,17,54112.0,1900.0,0,0,1.554236e+09,1.554237e+09,350.213117,17238,102
3,service_7,192.168.1.133,239.255.255.250,239.255.255.0/24,17,54577.0,1900.0,0,0,1.554236e+09,1.554236e+09,50.152472,2704,16
4,service_7,192.168.1.133,239.255.255.250,239.255.255.0/24,17,56581.0,1900.0,0,0,1.554236e+09,1.554236e+09,10.169791,1014,6


(891, 14)


In [12]:
from tqdm.auto import tqdm

pcap_files = sorted(BASE_DIR.glob("normal_*.pcap"))
print("num pcaps:", len(pcap_files))
print([p.name for p in pcap_files[:5]])

num pcaps: 13
['normal_1.pcap', 'normal_10.pcap', 'normal_11.pcap', 'normal_12.pcap', 'normal_13.pcap']


In [13]:
all_flow_dfs = []

for pcap_path in tqdm(pcap_files):
    stem = pcap_path.stem

    packet_csv_path = PACKET_DIR / f"{stem}_packets.csv"
    flow_csv_path   = FLOW_DIR / f"{stem}_meid20_flows.csv"

    print(f"\nProcessing {pcap_path.name} ...")

    # 1) pcap -> packet csv
    pcap_to_packet_csv(pcap_path, packet_csv_path)

    # 2) packet csv -> meid20 flow csv
    flow_df = packet_csv_to_meid20_flow_csv(packet_csv_path, flow_csv_path, ip_to_device)
    flow_df["pcap_name"] = pcap_path.name

    all_flow_dfs.append(flow_df)

    print(f"packet saved: {packet_csv_path}")
    print(f"flow saved:   {flow_csv_path}")
    print(f"flow rows:    {len(flow_df)}")

  0%|          | 0/13 [00:00<?, ?it/s]


Processing normal_1.pcap ...
packet saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/packet_csvs/normal_1_packets.csv
flow saved:   /content/drive/MyDrive/iot/iot device name/network/Meid20/flow_csvs/normal_1_meid20_flows.csv
flow rows:    899

Processing normal_10.pcap ...
packet saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/packet_csvs/normal_10_packets.csv
flow saved:   /content/drive/MyDrive/iot/iot device name/network/Meid20/flow_csvs/normal_10_meid20_flows.csv
flow rows:    687

Processing normal_11.pcap ...
packet saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/packet_csvs/normal_11_packets.csv
flow saved:   /content/drive/MyDrive/iot/iot device name/network/Meid20/flow_csvs/normal_11_meid20_flows.csv
flow rows:    493

Processing normal_12.pcap ...
packet saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/packet_csvs/normal_12_packets.csv
flow saved:   /content/drive/MyDrive/iot/iot device name/network/Meid20/fl

In [14]:
all_flows = pd.concat(all_flow_dfs, ignore_index=True)
all_flows = all_flows.sort_values("first_switched").reset_index(drop=True)

all_flow_path = MERGE_DIR / "meid20_features_all.csv"
all_flows.to_csv(all_flow_path, index=False)

print("saved merged file:", all_flow_path)
display(all_flows.head())
print(all_flows.shape)

saved merged file: /content/drive/MyDrive/iot/iot device name/network/Meid20/merged/meid20_features_all.csv


,device_name,src_ip,dst_ip,ipv4_dst_net,protocol,src_port,dst_port,src_tos,tcp_flags,first_switched,last_switched,flow_duration,in_bytes,in_pkts,pcap_name
0,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,2,152,1.554220e+09,1.554230e+09,9801.274541,11459206,36854,normal_1.pcap
1,service_2,192.168.1.193,239.255.255.250,239.255.255.0/24,17,54063.0,1900.0,0,0,1.554220e+09,1.554220e+09,2.980542,868,4,normal_1.pcap
2,service_3,192.168.1.190,216.239.38.10,216.239.38.0/24,17,20952.0,53.0,0,0,1.554220e+09,1.554220e+09,0.000000,100,1,normal_1.pcap
3,service_3,192.168.1.190,216.239.38.10,216.239.38.0/24,17,54663.0,53.0,0,0,1.554220e+09,1.554220e+09,0.000000,100,1,normal_1.pcap
4,service_3,192.168.1.190,91.189.91.139,91.189.91.0/24,17,59133.0,53.0,0,0,1.554220e+09,1.554220e+09,0.000000,89,1,normal_1.pcap


(16767, 15)


In [15]:
import ipaddress

def is_public_unicast_ipv4(ip):
    try:
        obj = ipaddress.ip_address(str(ip))
        return (
            obj.version == 4
            and obj.is_global
            and not obj.is_multicast
            and not obj.is_private
            and not obj.is_loopback
            and not obj.is_link_local
            and not obj.is_reserved
            and not obj.is_unspecified
        )
    except:
        return False

In [16]:
import pandas as pd
import numpy as np
import ipaddress
from functools import reduce
import operator

def to_num(x):
    return pd.to_numeric(x, errors="coerce")

def hex_or_dec_to_int(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    try:
        if s.startswith("0x") or s.startswith("0X"):
            return int(s, 16)
        return int(float(s))
    except:
        return np.nan

def ipv4_to_net24(ip):
    try:
        obj = ipaddress.ip_address(str(ip))
        if obj.version != 4:
            return "non_ipv4"
        parts = str(obj).split(".")
        return ".".join(parts[:3]) + ".0/24"
    except:
        return "unknown"

def is_public_unicast_ipv4(ip):
    try:
        obj = ipaddress.ip_address(str(ip))
        return (
            obj.version == 4
            and obj.is_global
            and not obj.is_multicast
            and not obj.is_private
            and not obj.is_loopback
            and not obj.is_link_local
            and not obj.is_reserved
            and not obj.is_unspecified
        )
    except:
        return False

def bitwise_or_series(s):
    vals = [int(v) for v in s.dropna().tolist()]
    if len(vals) == 0:
        return 0
    return reduce(operator.or_, vals, 0)

def assign_flow_segments(group, inactive_timeout=15, active_timeout=1800):
    group = group.sort_values("time").copy()

    seg_ids = []
    current_seg = 0
    seg_start_time = None
    prev_time = None

    for t in group["time"].tolist():
        start_new = False

        if prev_time is None:
            start_new = True
        else:
            if (t - prev_time) > inactive_timeout:
                start_new = True
            elif (t - seg_start_time) > active_timeout:
                start_new = True

        if start_new:
            current_seg += 1
            seg_start_time = t

        seg_ids.append(current_seg)
        prev_time = t

    group["flow_seg"] = seg_ids
    return group

def packet_csv_to_meid20_flow_csv(packet_csv_path, flow_csv_path, ip_to_device,
                                  inactive_timeout=15, active_timeout=1800):

    df = pd.read_csv(packet_csv_path, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "frame.time_epoch": "time",
        "ip.src": "src_ip",
        "ip.dst": "dst_ip",
        "ip.proto": "protocol",
        "tcp.srcport": "tcp_srcport",
        "tcp.dstport": "tcp_dstport",
        "udp.srcport": "udp_srcport",
        "udp.dstport": "udp_dstport",
        "frame.len": "frame_len",
        "tcp.flags": "tcp_flags",
        "ip.dsfield": "src_tos",
    }
    df = df.rename(columns=rename_map)

    df["time"] = df["time"].apply(to_num)
    df["protocol"] = df["protocol"].apply(to_num)
    df["frame_len"] = df["frame_len"].apply(to_num)
    df["tcp_flags"] = df["tcp_flags"].apply(hex_or_dec_to_int)
    df["src_tos"] = df["src_tos"].apply(hex_or_dec_to_int)

    for c in ["tcp_srcport", "tcp_dstport", "udp_srcport", "udp_dstport"]:
        df[c] = df[c].apply(to_num)

    df["src_port"] = df["tcp_srcport"].fillna(df["udp_srcport"])
    df["dst_port"] = df["tcp_dstport"].fillna(df["udp_dstport"])

    df = df.dropna(subset=["time", "src_ip", "dst_ip", "protocol", "frame_len"]).copy()

    df["src_ip"] = df["src_ip"].astype(str)
    df["dst_ip"] = df["dst_ip"].astype(str)

    # 只保留已知 IoT 设备发出的流
    df = df[df["src_ip"].isin(ip_to_device.keys())].copy()

    # 只保留真正公网单播目的地址
    df = df[df["dst_ip"].apply(is_public_unicast_ipv4)].copy()

    # 只保留有端口的 TCP/UDP 风格流
    df = df.dropna(subset=["src_port", "dst_port"]).copy()

    df["device_name"] = df["src_ip"].map(ip_to_device)
    df["ipv4_dst_net"] = df["dst_ip"].apply(ipv4_to_net24)

    base_keys = [
        "src_ip", "dst_ip", "protocol", "src_port", "dst_port",
        "device_name", "ipv4_dst_net"
    ]

    df = df.sort_values(base_keys + ["time"]).reset_index(drop=True)

    df = (
        df.groupby(base_keys, group_keys=False)
          .apply(lambda g: assign_flow_segments(
              g,
              inactive_timeout=inactive_timeout,
              active_timeout=active_timeout
          ))
          .reset_index(drop=True)
    )

    flow_keys = base_keys + ["flow_seg"]

    flow_df = (
        df.groupby(flow_keys, dropna=False)
          .agg(
              first_switched=("time", "min"),
              last_switched=("time", "max"),
              in_bytes=("frame_len", "sum"),
              in_pkts=("frame_len", "size"),
              src_tos=("src_tos", lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else 0),
              tcp_flags=("tcp_flags", bitwise_or_series),
          )
          .reset_index()
    )

    flow_df["flow_duration"] = flow_df["last_switched"] - flow_df["first_switched"]

    cols = [
        "device_name",
        "src_ip",
        "dst_ip",
        "ipv4_dst_net",
        "protocol",
        "src_port",
        "dst_port",
        "src_tos",
        "tcp_flags",
        "first_switched",
        "last_switched",
        "flow_duration",
        "in_bytes",
        "in_pkts",
    ]
    flow_df = flow_df[cols].copy()

    flow_df.to_csv(flow_csv_path, index=False)
    return flow_df

In [17]:
test_flow_csv = FLOW_DIR / "normal_2_meid20_flows_v2.csv"
flow_df = packet_csv_to_meid20_flow_csv(
    test_csv,
    test_flow_csv,
    ip_to_device,
    inactive_timeout=15,
    active_timeout=1800
)

print("saved:", test_flow_csv)
display(flow_df.head())
print(flow_df.shape)
print(flow_df["device_name"].value_counts())

/tmp/ipykernel_97438/310704402.py:143: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(


saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/flow_csvs/normal_2_meid20_flows_v2.csv


,device_name,src_ip,dst_ip,ipv4_dst_net,protocol,src_port,dst_port,src_tos,tcp_flags,first_switched,last_switched,flow_duration,in_bytes,in_pkts
0,service_1,192.168.1.152,13.32.126.20,13.32.126.0/24,6,48344.0,443.0,0,219,1.554232e+09,1.554233e+09,118.177476,3444,27
1,service_1,192.168.1.152,13.32.126.41,13.32.126.0/24,6,59576.0,443.0,0,219,1.554232e+09,1.554232e+09,5.711350,1362,10
2,service_1,192.168.1.152,13.32.126.41,13.32.126.0/24,6,59578.0,443.0,0,219,1.554232e+09,1.554232e+09,5.711621,1430,11
3,service_1,192.168.1.152,13.32.126.41,13.32.126.0/24,6,59580.0,443.0,0,219,1.554232e+09,1.554232e+09,5.710811,1430,11
4,service_1,192.168.1.152,13.32.126.41,13.32.126.0/24,6,59582.0,443.0,0,219,1.554232e+09,1.554232e+09,5.710890,1430,11


(864, 14)
device_name
service_3    753
service_1    111
Name: count, dtype: int64


In [18]:
flow_df[flow_df["dst_ip"].isin(["224.0.0.251", "239.255.255.250"])]
flow_df.sort_values("flow_duration", ascending=False).head(10)

,device_name,src_ip,dst_ip,ipv4_dst_net,protocol,src_port,dst_port,src_tos,tcp_flags,first_switched,last_switched,flow_duration,in_bytes,in_pkts
37,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,2,24,1.554235e+09,1.554237e+09,1799.951310,2119098,6966
38,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,2,24,1.554237e+09,1.554238e+09,1799.836698,2100424,6672
36,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,0,152,1.554233e+09,1.554235e+09,1799.820970,2165361,7165
34,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,2,152,1.554230e+09,1.554232e+09,1799.809974,2101332,6696
39,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,0,24,1.554238e+09,1.554240e+09,1799.619421,2108839,6779
35,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,2,24,1.554232e+09,1.554233e+09,1044.714976,1207243,3944
40,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,2,24,1.554240e+09,1.554241e+09,811.163741,972543,3345
0,service_1,192.168.1.152,13.32.126.20,13.32.126.0/24,6,48344.0,443.0,0,219,1.554232e+09,1.554233e+09,118.177476,3444,27
89,service_1,192.168.1.152,60.254.148.81,60.254.148.0/24,6,56012.0,80.0,0,219,1.554231e+09,1.554231e+09,115.111782,1460,17
8,service_1,192.168.1.152,13.35.146.89,13.35.146.0/24,6,45934.0,443.0,0,219,1.554231e+09,1.554231e+09,115.105423,3102,22


In [19]:
all_flow_dfs = []

for pcap_path in tqdm(pcap_files):
    stem = pcap_path.stem

    packet_csv_path = PACKET_DIR / f"{stem}_packets.csv"
    flow_csv_path   = FLOW_DIR / f"{stem}_meid20_flows_v2.csv"

    print(f"\nProcessing {pcap_path.name} ...")

    pcap_to_packet_csv(pcap_path, packet_csv_path)

    flow_df = packet_csv_to_meid20_flow_csv(
        packet_csv_path,
        flow_csv_path,
        ip_to_device,
        inactive_timeout=15,
        active_timeout=1800
    )

    flow_df["pcap_name"] = pcap_path.name
    all_flow_dfs.append(flow_df)

    print(f"flow rows: {len(flow_df)}")

  0%|          | 0/13 [00:00<?, ?it/s]


Processing normal_1.pcap ...


/tmp/ipykernel_97438/310704402.py:143: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(


flow rows: 855

Processing normal_10.pcap ...


/tmp/ipykernel_97438/310704402.py:143: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(


...

flow rows: 817

Processing normal_8.pcap ...


/tmp/ipykernel_97438/310704402.py:143: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(


flow rows: 1181

Processing normal_9.pcap ...


/tmp/ipykernel_97438/310704402.py:143: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(


flow rows: 1052


In [20]:
all_flows = pd.concat(all_flow_dfs, ignore_index=True)
all_flows = all_flows.sort_values("first_switched").reset_index(drop=True)

all_flow_path = MERGE_DIR / "meid20_features_all_v2.csv"
all_flows.to_csv(all_flow_path, index=False)

print("saved:", all_flow_path)
print(all_flows.shape)
display(all_flows.head())
print(all_flows["device_name"].value_counts())

saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/merged/meid20_features_all_v2.csv
(10010, 15)


,device_name,src_ip,dst_ip,ipv4_dst_net,protocol,src_port,dst_port,src_tos,tcp_flags,first_switched,last_switched,flow_duration,in_bytes,in_pkts,pcap_name
0,service_1,192.168.1.152,3.122.49.24,3.122.49.0/24,6,52976.0,1883.0,2,24,1.554220e+09,1.554222e+09,1799.978019,2110274,6812,normal_1.pcap
1,service_3,192.168.1.190,216.239.38.10,216.239.38.0/24,17,20952.0,53.0,0,0,1.554220e+09,1.554220e+09,0.000000,100,1,normal_1.pcap
2,service_3,192.168.1.190,216.239.38.10,216.239.38.0/24,17,54663.0,53.0,0,0,1.554220e+09,1.554220e+09,0.000000,100,1,normal_1.pcap
3,service_3,192.168.1.190,91.189.91.139,91.189.91.0/24,17,59133.0,53.0,0,0,1.554220e+09,1.554220e+09,0.000000,89,1,normal_1.pcap
4,service_3,192.168.1.190,205.251.197.191,205.251.197.0/24,17,29120.0,53.0,0,0,1.554220e+09,1.554220e+09,0.000000,100,1,normal_1.pcap


device_name
service_3    8571
service_1    1439
Name: count, dtype: int64


In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv(MERGE_DIR / "meid20_features_all_v2.csv")

df = df.dropna(subset=[
    "device_name", "ipv4_dst_net", "protocol", "src_port", "dst_port",
    "src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"
]).copy()

num_cols = ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

cat_cols = ["protocol", "src_port", "dst_port", "ipv4_dst_net"]
for c in cat_cols:
    df[c] = df[c].astype(str)

df = df.dropna().reset_index(drop=True)

print(df.shape)
print(df["device_name"].value_counts())

(10010, 15)
device_name
service_3    8571
service_1    1439
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import train_test_split

feature_cols_num = ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"]
feature_cols_cat = ["protocol", "src_port", "dst_port", "ipv4_dst_net"]
feature_cols = feature_cols_num + feature_cols_cat

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["device_name"]
)

print("train:", train_df.shape)
print("test :", test_df.shape)
print("\nTrain label counts:")
print(train_df["device_name"].value_counts())
print("\nTest label counts:")
print(test_df["device_name"].value_counts())

train: (8008, 15)
test : (2002, 15)

Train label counts:
device_name
service_3    6857
service_1    1151
Name: count, dtype: int64

Test label counts:
device_name
service_3    1714
service_1     288
Name: count, dtype: int64


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

X_train = train_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

preprocess = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), feature_cols_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
    ]
)

labels = sorted(df["device_name"].unique())
results = []

for lab in labels:
    y_train = (train_df["device_name"] == lab).astype(int)
    y_test  = (test_df["device_name"] == lab).astype(int)

    if y_train.sum() == 0 or y_test.sum() == 0:
        print(f"skip {lab}: no positive in train or test")
        continue

    clf = LGBMClassifier(
        objective="binary",
        n_estimators=60,
        num_leaves=128,
        random_state=42,
        n_jobs=-1
    )

    pipe = Pipeline([
        ("prep", preprocess),
        ("clf", clf)
    ])

    pipe.fit(X_train, y_train)
    y_score = pipe.predict_proba(X_test)[:, 1]
    ap = average_precision_score(y_test, y_score)

    results.append({
        "device_name": lab,
        "test_positive": int(y_test.sum()),
        "auprc": float(ap)
    })

res_df = pd.DataFrame(results).sort_values("auprc", ascending=False).reset_index(drop=True)
display(res_df)

macro_auprc = res_df["auprc"].mean()
print("Macro AUCPR =", macro_auprc)

[LightGBM] [Info] Number of positive: 1151, number of negative: 6857
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002398 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 709
[LightGBM] [Info] Number of data points in the train set: 8008, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.143731 -> initscore=-1.784639
[LightGBM] [Info] Start training from score -1.784639
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 6857, number of negative: 1151
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002654 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 709
[LightGBM] [Info] Number of data points in the train set: 8008, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.856269 -> initscore=1.784639
[LightGBM] [Info] Start training from score 1.784639
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,device_name,test_positive,auprc
0,service_3,1714,0.999299
1,service_1,288,0.996554


Macro AUCPR = 0.997926968593166


In [24]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier
import pandas as pd
import numpy as np

df = pd.read_csv(MERGE_DIR / "meid20_features_all_v2.csv")

df = df.dropna(subset=[
    "device_name", "ipv4_dst_net", "protocol", "src_port", "dst_port",
    "src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"
]).copy()

num_cols = ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

cat_cols = ["protocol", "src_port", "dst_port", "ipv4_dst_net"]
for c in cat_cols:
    df[c] = df[c].astype(str)

df = df.dropna().reset_index(drop=True)

feature_cols_num = ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"]
feature_cols_cat = ["protocol", "src_port", "dst_port", "ipv4_dst_net"]
feature_cols = feature_cols_num + feature_cols_cat

X = df[feature_cols].copy()
y_multi = df["device_name"].copy()

labels = sorted(y_multi.unique())

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold_id, (train_idx, test_idx) in enumerate(skf.split(X, y_multi), start=1):
    train_df = df.iloc[train_idx].copy()
    test_df  = df.iloc[test_idx].copy()

    X_train = train_df[feature_cols].copy()
    X_test  = test_df[feature_cols].copy()

    preprocess = ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), feature_cols_num),
            ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
        ]
    )

    for lab in labels:
        y_train = (train_df["device_name"] == lab).astype(int)
        y_test  = (test_df["device_name"] == lab).astype(int)

        if y_train.sum() == 0 or y_test.sum() == 0:
            continue

        clf = LGBMClassifier(
            objective="binary",
            n_estimators=60,
            num_leaves=128,
            random_state=42,
            n_jobs=-1
        )

        pipe = Pipeline([
            ("prep", preprocess),
            ("clf", clf)
        ])

        pipe.fit(X_train, y_train)
        y_score = pipe.predict_proba(X_test)[:, 1]
        ap = average_precision_score(y_test, y_score)

        fold_results.append({
            "fold": fold_id,
            "device_name": lab,
            "test_positive": int(y_test.sum()),
            "auprc": float(ap)
        })

cv_res_df = pd.DataFrame(fold_results)
display(cv_res_df.head())

device_mean_df = (
    cv_res_df.groupby("device_name", as_index=False)["auprc"]
             .mean()
             .sort_values("auprc", ascending=False)
             .reset_index(drop=True)
)

macro_auprc = device_mean_df["auprc"].mean()

print("\nPer-device mean AUCPR across 5 folds:")
display(device_mean_df)

print("Macro mean AUCPR across 5 folds =", macro_auprc)

[LightGBM] [Info] Number of positive: 1151, number of negative: 6857
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001610 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 710
[LightGBM] [Info] Number of data points in the train set: 8008, number of used features: 99
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.143731 -> initscore=-1.784639
[LightGBM] [Info] Start training from score -1.784639
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 6857, number of negative: 1151
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001523 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 710
[LightGBM] [Info] Number of data points in the train set: 8008, number of used features: 99
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.856269 -> initscore=1.784639
[LightGBM] [Info] Start training from score 1.784639
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1151, number of negative: 6857
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001837 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 724
[LightGBM] [Info] Number of data points in the train set: 8008, number of used features: 99
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.143731 -> initscore=-1.784639
[LightGBM] [Info] Start training from score -1.784639
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,fold,device_name,test_positive,auprc
0,1,service_1,288,0.999505
1,1,service_3,1714,0.999987
2,2,service_1,288,1.000000
3,2,service_3,1714,1.000000
4,3,service_1,288,0.993191



Per-device mean AUCPR across 5 folds:


,device_name,auprc
0,service_3,0.999611
1,service_1,0.998539


Macro mean AUCPR across 5 folds = 0.9990749673150452


In [30]:
import pandas as pd
import numpy as np
import ipaddress
from functools import reduce
import operator

def to_num(x):
    return pd.to_numeric(x, errors="coerce")

def hex_or_dec_to_int(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    try:
        if s.startswith("0x") or s.startswith("0X"):
            return int(s, 16)
        return int(float(s))
    except:
        return np.nan

def ipv4_to_net24(ip):
    try:
        obj = ipaddress.ip_address(str(ip))
        if obj.version != 4:
            return "non_ipv4"
        parts = str(obj).split(".")
        return ".".join(parts[:3]) + ".0/24"
    except:
        return "unknown"

def is_ipv4_unicast_keep_private(ip):
    try:
        obj = ipaddress.ip_address(str(ip))
        return (
            obj.version == 4
            and not obj.is_multicast
            and not obj.is_unspecified
            and not obj.is_loopback
            and str(obj) != "255.255.255.255"
        )
    except:
        return False

def bitwise_or_series(s):
    vals = [int(v) for v in s.dropna().tolist()]
    if len(vals) == 0:
        return 0
    return reduce(operator.or_, vals, 0)

def assign_flow_segments(group, inactive_timeout=15, active_timeout=1800):
    group = group.sort_values("time").copy()

    seg_ids = []
    current_seg = 0
    seg_start_time = None
    prev_time = None

    for t in group["time"].tolist():
        start_new = False

        if prev_time is None:
            start_new = True
        else:
            if (t - prev_time) > inactive_timeout:
                start_new = True
            elif (t - seg_start_time) > active_timeout:
                start_new = True

        if start_new:
            current_seg += 1
            seg_start_time = t

        seg_ids.append(current_seg)
        prev_time = t

    group["flow_seg"] = seg_ids
    return group

def packet_csv_to_devicecentric_flows(packet_csv_path, flow_csv_path, ip_to_device,
                                      inactive_timeout=15, active_timeout=1800):
    df = pd.read_csv(packet_csv_path, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "frame.time_epoch": "time",
        "ip.src": "src_ip",
        "ip.dst": "dst_ip",
        "ip.proto": "protocol",
        "tcp.srcport": "tcp_srcport",
        "tcp.dstport": "tcp_dstport",
        "udp.srcport": "udp_srcport",
        "udp.dstport": "udp_dstport",
        "frame.len": "frame_len",
        "tcp.flags": "tcp_flags",
        "ip.dsfield": "src_tos",
    }
    df = df.rename(columns=rename_map)

    df["time"] = df["time"].apply(to_num)
    df["protocol"] = df["protocol"].apply(to_num)
    df["frame_len"] = df["frame_len"].apply(to_num)
    df["tcp_flags"] = df["tcp_flags"].apply(hex_or_dec_to_int)
    df["src_tos"] = df["src_tos"].apply(hex_or_dec_to_int)

    for c in ["tcp_srcport", "tcp_dstport", "udp_srcport", "udp_dstport"]:
        df[c] = df[c].apply(to_num)

    # TCP/UDP 端口，缺失就填 0
    df["src_port"] = df["tcp_srcport"].fillna(df["udp_srcport"]).fillna(0)
    df["dst_port"] = df["tcp_dstport"].fillna(df["udp_dstport"]).fillna(0)

    df = df.dropna(subset=["time", "src_ip", "dst_ip", "protocol", "frame_len"]).copy()
    df["src_ip"] = df["src_ip"].astype(str)
    df["dst_ip"] = df["dst_ip"].astype(str)

    # 只保留 IPv4 单播，允许私网
    df = df[
        df["src_ip"].apply(is_ipv4_unicast_keep_private) &
        df["dst_ip"].apply(is_ipv4_unicast_keep_private)
    ].copy()

    # ---- 设备作为源：out 视角 ----
    out_view = df[df["src_ip"].isin(ip_to_device.keys())].copy()
    out_view["device_ip"]   = out_view["src_ip"]
    out_view["peer_ip"]     = out_view["dst_ip"]
    out_view["device_port"] = out_view["src_port"]
    out_view["peer_port"]   = out_view["dst_port"]
    out_view["direction"]   = "out"
    out_view["device_name"] = out_view["device_ip"].map(ip_to_device)

    # ---- 设备作为目的：in 视角 ----
    in_view = df[df["dst_ip"].isin(ip_to_device.keys())].copy()
    in_view["device_ip"]   = in_view["dst_ip"]
    in_view["peer_ip"]     = in_view["src_ip"]
    in_view["device_port"] = in_view["dst_port"]
    in_view["peer_port"]   = in_view["src_port"]
    in_view["direction"]   = "in"
    in_view["device_name"] = in_view["device_ip"].map(ip_to_device)

    # 合并两个视角
    view_df = pd.concat([out_view, in_view], ignore_index=True)

    # 只保留成功映射的设备
    view_df = view_df.dropna(subset=["device_name"]).copy()
    view_df["peer_net"] = view_df["peer_ip"].apply(ipv4_to_net24)

    base_keys = [
        "device_name", "device_ip", "peer_ip", "peer_net",
        "protocol", "device_port", "peer_port", "direction"
    ]

    view_df = view_df.sort_values(base_keys + ["time"]).reset_index(drop=True)

    view_df = (
        view_df.groupby(base_keys, group_keys=False)
               .apply(lambda g: assign_flow_segments(
                   g,
                   inactive_timeout=inactive_timeout,
                   active_timeout=active_timeout
               ))
               .reset_index(drop=True)
    )

    flow_keys = base_keys + ["flow_seg"]

    flow_df = (
        view_df.groupby(flow_keys, dropna=False)
               .agg(
                   first_switched=("time", "min"),
                   last_switched=("time", "max"),
                   in_bytes=("frame_len", "sum"),
                   in_pkts=("frame_len", "size"),
                   src_tos=("src_tos", lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else 0),
                   tcp_flags=("tcp_flags", bitwise_or_series),
               )
               .reset_index()
    )

    flow_df["flow_duration"] = flow_df["last_switched"] - flow_df["first_switched"]

    cols = [
        "device_name", "device_ip", "peer_ip", "peer_net",
        "direction", "protocol", "device_port", "peer_port",
        "src_tos", "tcp_flags",
        "first_switched", "last_switched", "flow_duration",
        "in_bytes", "in_pkts"
    ]
    flow_df = flow_df[cols].copy()

    flow_df.to_csv(flow_csv_path, index=False)
    return flow_df

In [31]:
test_flow_csv = FLOW_DIR / "normal_2_devicecentric_flows.csv"

flow_df = packet_csv_to_devicecentric_flows(
    test_csv,
    test_flow_csv,
    ip_to_device,
    inactive_timeout=15,
    active_timeout=1800
)

print("saved:", test_flow_csv)
display(flow_df.head())
print(flow_df.shape)
print(flow_df["device_name"].value_counts())

/tmp/ipykernel_97438/3186249029.py:158: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(


saved: /content/drive/MyDrive/iot/iot device name/network/Meid20/flow_csvs/normal_2_devicecentric_flows.csv


,device_name,device_ip,peer_ip,peer_net,direction,protocol,device_port,peer_port,src_tos,tcp_flags,first_switched,last_switched,flow_duration,in_bytes,in_pkts
0,service_1,192.168.1.152,13.32.126.20,13.32.126.0/24,in,6,48344.0,443.0,0,91,1.554232e+09,1.554233e+09,118.069186,28930,26
1,service_1,192.168.1.152,13.32.126.20,13.32.126.0/24,out,6,48344.0,443.0,0,219,1.554232e+09,1.554233e+09,118.177476,3444,27
2,service_1,192.168.1.152,13.32.126.41,13.32.126.0/24,in,6,59576.0,443.0,0,91,1.554232e+09,1.554232e+09,5.745095,4274,8
3,service_1,192.168.1.152,13.32.126.41,13.32.126.0/24,out,6,59576.0,443.0,0,219,1.554232e+09,1.554232e+09,5.711350,1362,10
4,service_1,192.168.1.152,13.32.126.41,13.32.126.0/24,in,6,59578.0,443.0,0,91,1.554232e+09,1.554232e+09,5.745467,4274,8


(6335, 15)
device_name
service_3    3312
service_1    2374
service_2     624
service_4      25
Name: count, dtype: int64


In [32]:
from tqdm.auto import tqdm
all_flow_dfs = []

for pcap_path in tqdm(pcap_files):
    stem = pcap_path.stem

    packet_csv_path = PACKET_DIR / f"{stem}_packets.csv"
    flow_csv_path   = FLOW_DIR / f"{stem}_devicecentric_flows.csv"

    pcap_to_packet_csv(pcap_path, packet_csv_path)

    flow_df = packet_csv_to_devicecentric_flows(
        packet_csv_path,
        flow_csv_path,
        ip_to_device,
        inactive_timeout=15,
        active_timeout=1800
    )

    flow_df["pcap_name"] = pcap_path.name
    all_flow_dfs.append(flow_df)

all_flows = pd.concat(all_flow_dfs, ignore_index=True)
all_flows = all_flows.sort_values("first_switched").reset_index(drop=True)

all_flow_path = MERGE_DIR / "meid20_features_all_devicecentric.csv"
all_flows.to_csv(all_flow_path, index=False)

print(all_flows.shape)
print(all_flows["device_name"].value_counts())
display(all_flows.head())

  0%|          | 0/13 [00:00<?, ?it/s]

/tmp/ipykernel_97438/3186249029.py:158: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(
/tmp/ipykernel_97438/3186249029.py:158: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: assign_flow_segments(
/tmp/ipykernel_97438/3186249029.py:158: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a futur

(77336, 16)
device_name
service_3    40503
service_1    26299
service_4     6785
service_2     3703
service_5       46
Name: count, dtype: int64


,device_name,device_ip,peer_ip,peer_net,direction,protocol,device_port,peer_port,src_tos,tcp_flags,first_switched,last_switched,flow_duration,in_bytes,in_pkts,pcap_name
0,service_1,192.168.1.152,192.168.1.192,192.168.1.0/24,out,6,1880.0,40571.0,0,24,1.554220e+09,1.554222e+09,1799.992882,10279445,15536,normal_1.pcap
1,service_2,192.168.1.192,192.168.1.152,192.168.1.0/24,in,6,40571.0,1880.0,0,24,1.554220e+09,1.554222e+09,1799.992882,10279445,15536,normal_1.pcap
2,service_3,192.168.1.190,192.168.1.152,192.168.1.0/24,in,6,43539.0,1880.0,0,24,1.554220e+09,1.554222e+09,1799.992661,9964623,14806,normal_1.pcap
3,service_1,192.168.1.152,192.168.1.190,192.168.1.0/24,out,6,1880.0,43539.0,0,24,1.554220e+09,1.554222e+09,1799.992661,9964623,14806,normal_1.pcap
4,service_3,192.168.1.190,192.168.1.152,192.168.1.0/24,out,6,43539.0,1880.0,0,16,1.554220e+09,1.554222e+09,1799.992634,1001788,14732,normal_1.pcap


In [33]:
df = pd.read_csv(MERGE_DIR / "meid20_features_all_devicecentric.csv")

df = df.dropna(subset=[
    "device_name", "peer_net", "direction", "protocol",
    "device_port", "peer_port", "src_tos", "tcp_flags",
    "flow_duration", "in_bytes", "in_pkts"
]).copy()

num_cols = ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

cat_cols = ["direction", "protocol", "device_port", "peer_port", "peer_net"]
for c in cat_cols:
    df[c] = df[c].astype(str)

df = df.dropna().reset_index(drop=True)

print(df.shape)
print(df["device_name"].value_counts())

(77336, 16)
device_name
service_3    40503
service_1    26299
service_4     6785
service_2     3703
service_5       46
Name: count, dtype: int64


In [34]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier
import pandas as pd

feature_cols_num = ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"]
feature_cols_cat = ["direction", "protocol", "device_port", "peer_port", "peer_net"]
feature_cols = feature_cols_num + feature_cols_cat

X = df[feature_cols].copy()
y_multi = df["device_name"].copy()

labels = sorted(y_multi.unique())

min_count = y_multi.value_counts().min()
n_splits = min(5, int(min_count))
if n_splits < 2:
    raise ValueError("至少有一个 service 样本太少，无法做分层交叉验证。")

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

fold_results = []

for fold_id, (train_idx, test_idx) in enumerate(skf.split(X, y_multi), start=1):
    train_df = df.iloc[train_idx].copy()
    test_df  = df.iloc[test_idx].copy()

    X_train = train_df[feature_cols].copy()
    X_test  = test_df[feature_cols].copy()

    preprocess = ColumnTransformer(
        transformers=[
            ("num", MinMaxScaler(), feature_cols_num),
            ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
        ]
    )

    for lab in labels:
        y_train = (train_df["device_name"] == lab).astype(int)
        y_test  = (test_df["device_name"] == lab).astype(int)

        clf = LGBMClassifier(
            objective="binary",
            n_estimators=60,
            num_leaves=128,
            random_state=42,
            n_jobs=-1
        )

        pipe = Pipeline([
            ("prep", preprocess),
            ("clf", clf)
        ])

        pipe.fit(X_train, y_train)
        y_score = pipe.predict_proba(X_test)[:, 1]
        ap = average_precision_score(y_test, y_score)

        fold_results.append({
            "fold": fold_id,
            "device_name": lab,
            "test_positive": int(y_test.sum()),
            "auprc": float(ap)
        })

cv_res_df = pd.DataFrame(fold_results)

device_mean_df = (
    cv_res_df.groupby("device_name", as_index=False)["auprc"]
             .mean()
             .sort_values("auprc", ascending=False)
             .reset_index(drop=True)
)

macro_auprc = device_mean_df["auprc"].mean()

display(device_mean_df)
print("Macro mean AUCPR =", macro_auprc)

[LightGBM] [Info] Number of positive: 21039, number of negative: 40829
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.492504 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1087
[LightGBM] [Info] Number of data points in the train set: 61868, number of used features: 154
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.340063 -> initscore=-0.663015
[LightGBM] [Info] Start training from score -0.663015
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 2962, number of negative: 58906
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.048115 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1087
[LightGBM] [Info] Number of data points in the train set: 61868, number of used features: 154
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047876 -> initscore=-2.990078
[LightGBM] [Info] Start training from score -2.990078
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 32403, number of negative: 29465
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.064533 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1087
[LightGBM] [Info] Number of data points in the train set: 61868, number of used features: 154
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523744 -> initscore=0.095048
[LightGBM] [Info] Start training from score 0.095048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 37, number of negative: 61832
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.068424 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1082
[LightGBM] [Info] Number of data points in the train set: 61869, number of used features: 151
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000598 -> initscore=-7.421258
[LightGBM] [Info] Start training from score -7.421258
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,device_name,auprc
0,service_4,1.000000
1,service_3,0.999953
2,service_1,0.999919
3,service_2,0.993177
4,service_5,0.151877


Macro mean AUCPR = 0.8289851716696038


In [62]:
import pandas as pd
import numpy as np

df = pd.read_csv(MERGE_DIR / "meid20_features_all_devicecentric.csv")

df = df.dropna(subset=[
    "device_name", "direction", "protocol",
    "src_tos", "tcp_flags", "flow_duration",
    "in_bytes", "in_pkts", "first_switched"
]).copy()

for c in ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts", "first_switched"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna().copy()
df = df.sort_values("first_switched").reset_index(drop=True)

# 不换时区，直接按抓包 epoch 解码
df["dt"] = pd.to_datetime(df["first_switched"], unit="s")
df["hour"] = df["dt"].dt.hour

# 你的定义：02:00–06:00 视作 idle
df["mode"] = np.where(df["hour"].between(2, 5), "idle", "active")

print(df.shape)
print(df["device_name"].value_counts())
print(df["mode"].value_counts())

(77336, 19)
device_name
service_3    40503
service_1    26299
service_4     6785
service_2     3703
service_5       46
Name: count, dtype: int64
mode
active    61863
idle      15473
Name: count, dtype: int64


In [63]:
WINDOW_SEC = 1800   # 30 min
GAP_SEC = 3600      # 1 hour gap

df["block_start"] = (df["first_switched"] // WINDOW_SEC) * WINDOW_SEC

In [64]:
all_services = sorted(df["device_name"].unique())

def split_one_mode_by_blocks(df_mode, test_ratio=0.2, gap_sec=3600):
    blocks = sorted(df_mode["block_start"].unique())
    if len(blocks) < 2:
        raise ValueError("块数太少，无法切分")

    n_test = max(1, int(round(len(blocks) * test_ratio)))

    # 从较小 test 块数开始往上试，直到 train/test 都覆盖全部 service
    for k in range(n_test, len(blocks)):
        test_blocks = blocks[-k:]
        test_start = min(test_blocks)

        train_part = df_mode[df_mode["block_start"] < test_start - gap_sec].copy()
        test_part  = df_mode[df_mode["block_start"].isin(test_blocks)].copy()

        train_services = set(train_part["device_name"].unique())
        test_services  = set(test_part["device_name"].unique())

        if set(all_services).issubset(train_services) and set(all_services).issubset(test_services):
            return train_part, test_part, test_blocks

    raise ValueError("当前 mode 下找不到同时覆盖全部 service 的连续时间块切分")

idle_df = df[df["mode"] == "idle"].copy()
active_df = df[df["mode"] == "active"].copy()

idle_train, idle_test, idle_test_blocks = split_one_mode_by_blocks(idle_df, test_ratio=0.2, gap_sec=GAP_SEC)
active_train, active_test, active_test_blocks = split_one_mode_by_blocks(active_df, test_ratio=0.2, gap_sec=GAP_SEC)

train_mix = pd.concat([idle_train, active_train], ignore_index=True)
test_idle = idle_test.copy()
test_active = active_test.copy()
test_mix = pd.concat([idle_test, active_test], ignore_index=True)

print("train_mix :", train_mix.shape)
print("test_idle :", test_idle.shape)
print("test_active:", test_active.shape)
print("test_mix  :", test_mix.shape)

print("\nTrain label counts:")
print(train_mix["device_name"].value_counts())

print("\nTest idle label counts:")
print(test_idle["device_name"].value_counts())

print("\nTest active label counts:")
print(test_active["device_name"].value_counts())

train_mix : (34975, 20)
test_idle : (7461, 20)
test_active: (30187, 20)
test_mix  : (37648, 20)

Train label counts:
device_name
service_3    16627
service_1    12329
service_2     3151
service_4     2833
service_5       35
Name: count, dtype: int64

Test idle label counts:
device_name
service_3    4130
service_1    2544
service_4     693
service_2      91
service_5       3
Name: count, dtype: int64

Test active label counts:
device_name
service_3    17976
service_1    10005
service_4     2114
service_2       85
service_5        7
Name: count, dtype: int64


In [65]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

# 行为优先特征
feature_cols_num = ["src_tos", "tcp_flags", "flow_duration", "in_bytes", "in_pkts"]
feature_cols_cat = ["direction", "protocol"]
feature_cols = feature_cols_num + feature_cols_cat

X_train = train_mix[feature_cols].copy()
labels = sorted(train_mix["device_name"].unique())

preprocess = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), feature_cols_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
    ]
)

models = {}

for lab in labels:
    y_train = (train_mix["device_name"] == lab).astype(int)

    clf = LGBMClassifier(
        objective="binary",
        n_estimators=60,
        num_leaves=128,
        random_state=42,
        n_jobs=-1
    )

    pipe = Pipeline([
        ("prep", preprocess),
        ("clf", clf)
    ])

    pipe.fit(X_train, y_train)
    models[lab] = pipe

print("trained labels:", labels)

[LightGBM] [Info] Number of positive: 12329, number of negative: 22646
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006003 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 798
[LightGBM] [Info] Number of data points in the train set: 34975, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.352509 -> initscore=-0.608029
[LightGBM] [Info] Start training from score -0.608029
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [66]:
from sklearn.metrics import average_precision_score

def evaluate_scenario(test_df, scenario_name, models, feature_cols):
    X_test = test_df[feature_cols].copy()
    rows = []

    for lab, pipe in models.items():
        y_test = (test_df["device_name"] == lab).astype(int)

        if y_test.sum() == 0:
            continue

        y_score = pipe.predict_proba(X_test)[:, 1]
        ap = average_precision_score(y_test, y_score)

        rows.append({
            "scenario": scenario_name,
            "device_name": lab,
            "test_positive": int(y_test.sum()),
            "auprc": float(ap)
        })

    res = pd.DataFrame(rows).sort_values("auprc", ascending=False).reset_index(drop=True)
    macro = res["auprc"].mean() if len(res) > 0 else np.nan
    return res, macro

idle_res, idle_macro = evaluate_scenario(test_idle, "Test: Idle", models, feature_cols)
active_res, active_macro = evaluate_scenario(test_active, "Test: Active", models, feature_cols)
mix_res, mix_macro = evaluate_scenario(test_mix, "Test: Mix", models, feature_cols)

print("Idle macro AUCPR  =", idle_macro)
print("Active macro AUCPR=", active_macro)
print("Mix macro AUCPR   =", mix_macro)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

Idle macro AUCPR  = 0.7960418500919721
Active macro AUCPR= 0.7658011789386958
Mix macro AUCPR   = 0.7800548101748871


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
